Step 1: Data Preprocessing

In [1]:
import pandas as pd
import numpy as np
import torch
import random
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, TensorDataset

# Set a fixed random seed for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

def create_8x8_matrix(data_row):
    numeric_row = pd.to_numeric(data_row, errors='coerce').fillna(0)
    num_elements_required = 64
    if len(numeric_row) < num_elements_required:
        numeric_row = np.pad(numeric_row, (0, num_elements_required - len(numeric_row)), 'constant')
    matrix = np.array(numeric_row).reshape(8, 8)
    return matrix

def process_dataset(dataset):
    matrices = []
    for _, row in dataset.iterrows():
        matrix = create_8x8_matrix(row)
        matrices.append(matrix)
    return matrices

# Load dataset
df = pd.read_csv('kddcup.data_10_percent.csv', header=None)

# Identify and remove non-numeric columns
non_numeric_columns = [1, 2, 3]
df_features = df.drop(non_numeric_columns, axis=1)

# Process each row to create 8x8 matrices
matrices = process_dataset(df_features)

# Convert to a 3D numpy array
X = np.array(matrices)

# Encode the labels
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df.iloc[:, -1])

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=SEED)

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)


Step 2: Building the PDAE Model

In [ ]:
import torch
import torch.nn as nn

KERNEL = 3
HIDDEN = 64   # linear layer width before the conv layer
LATENT = 32   # latent size per branch


class ConvEncoder(nn.Module):
    def __init__(self, n_in, dilation, dropout, latent=LATENT):
        super().__init__()
        conv_len = HIDDEN - dilation * (KERNEL - 1)
        self.net = nn.Sequential(
            nn.Linear(n_in, HIDDEN),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(1, 1, kernel_size=KERNEL, stride=1, dilation=dilation),
            nn.Flatten(),
            nn.Linear(conv_len, latent),
        )

    def forward(self, x):
        return self.net(x.unsqueeze(1))


class ConvDecoder(nn.Module):
    def __init__(self, n_out, dilation, dropout, latent=LATENT):
        super().__init__()
        mid = n_out - dilation * (KERNEL - 1)
        self.net = nn.Sequential(
            nn.Linear(latent, mid),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Unflatten(dim=1, unflattened_size=(1, mid)),
            nn.ConvTranspose1d(1, 1, kernel_size=KERNEL, stride=1, dilation=dilation),
            nn.Flatten(),
            nn.Linear(n_out, n_out),
        )

    def forward(self, z):
        return self.net(z)


class SharedDecoder(nn.Module):
    def __init__(self, n_out, in_dim, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, n_out),
        )

    def forward(self, z):
        return self.net(z)


class DeepAutoencoder(nn.Module):
    def __init__(self, num_input_features, num_classes, dilations=(1,), dropout_rate=0.4):
        super().__init__()
        self.num_input_features = num_input_features
        self.dilations = tuple(dilations)
        self.encoders = nn.ModuleList(
            [ConvEncoder(num_input_features, d, dropout_rate) for d in self.dilations]
        )
        latent_total = LATENT * len(self.dilations)
        if len(self.dilations) == 1:
            self.decoder = ConvDecoder(num_input_features, self.dilations[0], dropout_rate)
        else:
            self.decoder = SharedDecoder(num_input_features, latent_total, dropout_rate)
        self.classifier = nn.Linear(latent_total, num_classes)

    def forward(self, x):
        x = x.view(-1, self.num_input_features)
        z = torch.cat([enc(x) for enc in self.encoders], dim=1)
        return self.decoder(z), self.classifier(z)


# ---- ----
DILATIONS = (1,2,3)            
num_input_features = 64     
# ---------------------------------------------

num_classes = len(label_encoder.classes_)
print(num_classes)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DeepAutoencoder(num_input_features, num_classes, dilations=DILATIONS).to(device)

Step 3: Training the Model

In [3]:
import torch.optim as optim  # Import the optim module

LEARNING_RATE = 0.001  # Adjusted learning rate
BATCH_SIZE = 64        # Adjusted batch size

criterion_reconstruction = nn.L1Loss()
criterion_classification = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

train_data = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)

num_epochs = 100



for epoch in range(num_epochs):
    model.train()
    running_loss_reconstruction = 0.0
    running_loss_classification = 0.0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        decoded, classification = model(inputs.view(-1, num_input_features))
        loss_reconstruction = criterion_reconstruction(decoded, inputs.view(-1, num_input_features))
        loss_classification = criterion_classification(classification, labels)
        loss = loss_reconstruction + loss_classification
        loss.backward()
        optimizer.step()

        running_loss_reconstruction += loss_reconstruction.item()
        running_loss_classification += loss_classification.item()

    print(f'Epoch [{epoch+1}/{num_epochs}], Reconstruction Loss: {running_loss_reconstruction / len(train_loader)}, Classification Loss: {running_loss_classification / len(train_loader)}')

Epoch [1/100], Reconstruction Loss: 85.29672740349909, Classification Loss: 380.19700493421607
Epoch [2/100], Reconstruction Loss: 84.9473298881544, Classification Loss: 128.1321983933952
Epoch [3/100], Reconstruction Loss: 84.65796136938434, Classification Loss: 544.0812630884817
Epoch [4/100], Reconstruction Loss: 84.38708630761128, Classification Loss: 156.3269443522381
Epoch [5/100], Reconstruction Loss: 84.14039833879224, Classification Loss: 331.9763106718311
Epoch [6/100], Reconstruction Loss: 83.90098620070489, Classification Loss: 278.4835190787859
Epoch [7/100], Reconstruction Loss: 83.66575456629145, Classification Loss: 220.5294380535243
Epoch [8/100], Reconstruction Loss: 83.43343568581167, Classification Loss: 125.53581197136391
Epoch [9/100], Reconstruction Loss: 83.20317529866115, Classification Loss: 52.924489058761445
Epoch [10/100], Reconstruction Loss: 82.97650370919025, Classification Loss: 33.6486381449753
Epoch [11/100], Reconstruction Loss: 82.74859036789863, Cl

Step 4: Evaluating the Model and Calculating Metrics

In [4]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import csv

model.eval()
true_labels = []
predicted_labels = []

test_data = TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.view(-1, num_input_features)
        inputs, labels = inputs.to(device), labels.to(device)
        decoded, classification = model(inputs)
        _, predicted = torch.max(classification, 1)
        true_labels.extend(labels.cpu().tolist())
        predicted_labels.extend(predicted.cpu().tolist())

accuracy = accuracy_score(true_labels, predicted_labels)
precision = precision_score(true_labels, predicted_labels, average='weighted', zero_division=1)
recall = recall_score(true_labels, predicted_labels, average='weighted', zero_division=1)
f1 = f1_score(true_labels, predicted_labels, average='weighted', zero_division=1)
conf_matrix = confusion_matrix(true_labels, predicted_labels)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print("Confusion Matrix:\n", conf_matrix)

csv_file = "kdd99d3d2.csv"

# نوشتن داده به فایل CSV
with open(csv_file, mode='w', newline='', encoding='utf-8-sig') as file:
    writer = csv.writer(file)
    writer.writerows(conf_matrix)

print(f"فایل {csv_file} با موفقیت ایجاد شد.")

Accuracy: 0.9973442585785306
Precision: 0.9973077350150366
Recall: 0.9973442585785306
F1 Score: 0.9970294565770004
Confusion Matrix:
 [[  536     0     0     0     0     0     0     0     0     0     0     5
      0     0     0     0     0     0     0     0     0]
 [    3     0     0     0     0     0     0     0     0     0     0     8
      0     0     0     0     0     0     0     0     0]
 [    0     0     0     0     0     0     0     0     0     0     0     1
      0     0     0     0     0     0     0     0     0]
 [    0     0     0     0     0     0     0     0     0     0     0    10
      0     0     0     0     0     0     0     0     0]
 [    0     0     0     0     2     0     0     0     0     0     0     1
      0     0     0     0     0     0     0     0     0]
 [    0     0     0     0     0   315     0     0     0     0     0     9
      0     0     0     0     0     0     0     0     0]
 [    0     0     0     0     0     0     5     0     0     0     0     0
      